#Setup

In [ ]:
%cd /content/drive/MyDrive/263

/content/drive/MyDrive/263


In [ ]:
from google.colab import userdata
hf_token = userdata.get('hugging_face')
openai_token = userdata.get('openai')

In [ ]:
#import dataset
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/263/CS263_dataset.csv')
df.head()

,id,category,interaction,culture,label,confidence_score,cultural_context
0,SAC001,family,User: I didn’t tell my parents I moved to anot...,"US, adult independence norm",acceptable,0.86,Adult autonomy and privacy are commonly respec...
1,SAC002,family,User: I didn’t tell my parents I moved to anot...,"China, family-involvement norm",context-dependent,0.78,Major life decisions are often shared with fam...
2,SAC003,family,User: I skipped a family dinner because I want...,"US, personal-boundary norm",acceptable,0.84,Personal time and emotional boundaries are gen...
3,SAC004,family,User: I skipped a family dinner because I want...,"Italy, family-gathering norm",context-dependent,0.76,"Family gatherings can be socially important, b..."
4,SAC005,family,User: I didn’t help my sibling financially.\nL...,"US, financial-independence norm",context-dependent,0.80,Individual financial responsibility is emphasi...


#DeBERTa 2 Class + Threshold
We chose DeBERTa as one of our base models because its a model that is optimized for Natural Language Inference, trained to handle entailment-style reasoning.

**Design**

Have DeBERTa predict between "acceptable" and "not acceptable", if the confidence is below a threshold, return "context-dependent" instead.

In [ ]:
import re
import pandas as pd
from transformers import pipeline

# 1. Load dataset
df = pd.read_csv('CS263_dataset.csv')

# 2. Parse User sentence
def parse_user_interaction(text):
    match = re.search(r"User:\s*(.*?)(?:\\n|\n|$)", text)
    return match.group(1).strip() if match else None

df["situation"] = df["interaction"].apply(parse_user_interaction)

# 3. Load DeBERTa MNLI model
classifier = pipeline(
    "zero-shot-classification",
    model="microsoft/deberta-large-mnli",
    token=hf_token
)

candidate_labels = ["acceptable behavior", "unacceptable behavior"]

def predict_acceptability(situation, threshold=0.65):
    result = classifier(
        situation,
        candidate_labels=[
            "acceptable behavior",
            "not acceptable behavior"
        ],
        hypothesis_template="This behavior is {}."
    )

    top_label = result["labels"][0]
    top_score = result["scores"][0]

    if top_score < threshold:
        return "context-dependent"

    if top_label == "acceptable behavior":
        return "acceptable"
    else:
        return "not acceptable"

# 5. Predict
predictions = []

for _, row in df.iterrows():

    pred_label = predict_acceptability(row["situation"])

    predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

# 6. Create result DataFrame
results_df = pd.DataFrame(predictions)

results_df.head()
results_df.to_csv("deberta_predictions.csv", index=False)

Loading weights:   0%|          | 0/392 [00:00<?, ?it/s]

Exception in thread Thread-auto_conversion:
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.12/threading.py", line 1012, in run
    self._target(*self._args, **self._kwargs)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 116, in auto_conversion
    raise e
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 95, in auto_conversion
    sha = get_conversion_pr_reference(api, pretrained_model_name_or_path, **cached_file_kwargs)
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 71, in get_conversion_pr_reference
    spawn_conversion(token, private, model_id)
  File "/usr/local/lib/python3.12/dist-packages/transformers/safetensors_conversion.py", line 48, in spawn_con

In [ ]:
!pip install openai

#ChatGPT

In [ ]:
from openai import OpenAI
import json
import pandas as pd

client = OpenAI(api_key=openai_token)

def predict_acceptability_gpt(situation):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0,
        messages=[
            {
                "role": "system",
                "content": """
You are a social acceptability classifier.

Classify the user's situation into exactly one label:
- acceptable
- not acceptable
- context-dependent

Do not use cultural context unless it is explicitly included in the situation.
Return only JSON in this format:
{"label": "..."}
"""
            },
            {
                "role": "user",
                "content": f"Situation: {situation}"
            }
        ]
    )

    text = response.choices[0].message.content
    return json.loads(text)["label"]

In [ ]:
gpt_predictions = []

for _, row in df.iterrows():
    pred_label = predict_acceptability_gpt(row["situation"])

    gpt_predictions.append({
        "id": row["id"],
        "situation": row["situation"],
        "gold_label": row["label"],
        "prediction_label": pred_label
    })

gpt_results_df = pd.DataFrame(gpt_predictions)

gpt_results_df.head()
gpt_results_df.to_csv("gpt_predictions.csv", index=False)

#Accuracy Analysis

In [ ]:
from sklearn.metrics import accuracy_score

# DeBERTa
deberta_acc = accuracy_score(
    results_df["gold_label"],
    results_df["prediction_label"]
)

# GPT
gpt_acc = accuracy_score(
    gpt_results_df["gold_label"],
    gpt_results_df["prediction_label"]
)

print(f"DeBERTa Accuracy: {deberta_acc:.4f}")
print(f"GPT Accuracy: {gpt_acc:.4f}")

DeBERTa Accuracy: 0.4167
GPT Accuracy: 0.5750


In [ ]:
with open("accuracy_summary.txt", "w") as f:
    f.write(f"DeBERTa Accuracy: {deberta_acc:.4f}\n")
    f.write(f"GPT Accuracy: {gpt_acc:.4f}\n")

#Evaluation

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

labels = ["acceptable", "unacceptable", "context-dependent"]

def evaluate_model(df, model_name, save_path):
    y_true = df["gold_label"]
    y_pred = df["prediction_label"]

    # Accuracy
    acc = accuracy_score(y_true, y_pred)

    # Classification report
    report = classification_report(y_true, y_pred, labels=labels)

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred, labels=labels)

    # Print results
    print(f"\n===== {model_name} =====")
    print(f"Accuracy: {acc:.4f}\n")
    print("Classification Report:")
    print(report)
    print("Confusion Matrix:")
    print(cm)

    # Save to file
    with open(save_path, "w") as f:
        f.write(f"===== {model_name} =====\n")
        f.write(f"Accuracy: {acc:.4f}\n\n")
        f.write("Classification Report:\n")
        f.write(report + "\n")
        f.write("Confusion Matrix:\n")
        f.write(str(cm))

# Run evaluations
evaluate_model(results_df, "DeBERTa", "deberta_eval.txt")
evaluate_model(gpt_results_df, "GPT", "gpt_eval.txt")


===== DeBERTa =====
Accuracy: 0.4167

Classification Report:
                   precision    recall  f1-score   support

       acceptable       0.36      0.22      0.27        23
     unacceptable       0.00      0.00      0.00         0
context-dependent       0.38      0.28      0.32        47

        micro avg       0.38      0.26      0.31        70
        macro avg       0.25      0.16      0.20        70
     weighted avg       0.37      0.26      0.30        70

Confusion Matrix:
[[ 5  0  6]
 [ 0  0  0]
 [ 6  0 13]]

===== GPT =====
Accuracy: 0.5750

Classification Report:
                   precision    recall  f1-score   support

       acceptable       0.50      0.74      0.60        23
     unacceptable       0.00      0.00      0.00         0
context-dependent       0.50      0.28      0.36        47

        micro avg       0.50      0.43      0.46        70
        macro avg       0.33      0.34      0.32        70
     weighted avg       0.50      0.43      0.44     

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in labels with no true nor predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/me

Aggregate Result

In [ ]:
import pandas as pd

df = pd.read_csv("CS263_dataset.csv")

deberta_df = pd.read_csv("deberta_predictions.csv")
gpt_df = pd.read_csv("gpt_predictions.csv")

deberta_df = deberta_df.rename(columns={"prediction_label": "deberta_label"})
gpt_df = gpt_df.rename(columns={"prediction_label": "gpt_label"})

df = df.merge(
    deberta_df[["id", "situation", "deberta_label"]],
    on="id",
    how="left"
)

df = df.merge(
    gpt_df[["id", "gpt_label"]],
    on="id",
    how="left"
)

df.to_csv("CS263_dataset_with_predictions.csv", index=False)

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# ===== 1. Load CSV =====
input_file = "CS263_dataset_with_predictions.csv"
df = pd.read_csv(input_file)

# ===== 2. Extract country from culture column =====
# Example: "US, adult independence norm" -> "US"
df["country"] = df["culture"].str.split(",").str[0].str.strip()

# ===== 3. Normalize labels =====
label_cols = ["label", "deberta_label", "gpt_label"]

for col in label_cols:
    df[col] = df[col].astype(str).str.lower().str.strip()

# ===== 4. Accuracy grouped by country =====
summary_rows = []

for country, group in df.groupby("country"):
    deberta_acc = accuracy_score(group["label"], group["deberta_label"])
    gpt_acc = accuracy_score(group["label"], group["gpt_label"])

    summary_rows.append({
        "country": country,
        "n_samples": len(group),
        "deberta_accuracy": deberta_acc,
        "gpt_accuracy": gpt_acc
    })

summary_df = pd.DataFrame(summary_rows).sort_values("country")

print("\n=== Accuracy by Country ===")
print(summary_df)

# ===== 5. Overall accuracy =====
overall = pd.DataFrame([
    {
        "model": "DeBERTa",
        "accuracy": accuracy_score(df["label"], df["deberta_label"])
    },
    {
        "model": "GPT",
        "accuracy": accuracy_score(df["label"], df["gpt_label"])
    }
])

print("\n=== Overall Accuracy ===")
print(overall)

# ===== 6. Full classification report by country =====
for country, group in df.groupby("country"):
    print(f"\n\n================ {country} ================")

    print("\n--- DeBERTa Classification Report ---")
    print(classification_report(
        group["label"],
        group["deberta_label"],
        zero_division=0
    ))

    print("\n--- GPT Classification Report ---")
    print(classification_report(
        group["label"],
        group["gpt_label"],
        zero_division=0
    ))

# ===== 7. Save country-level summary =====
summary_df.to_csv("accuracy_by_country.csv", index=False)
overall.to_csv("overall_accuracy.csv", index=False)

print("\nSaved:")
print("- accuracy_by_country.csv")
print("- overall_accuracy.csv")


=== Accuracy by Country ===
        country  n_samples  deberta_accuracy  gpt_accuracy
0        Brazil          4          0.000000      0.500000
1         China         13          0.538462      0.307692
2       Finland          1          0.000000      1.000000
3        France          2          0.500000      1.000000
4       Germany         12          0.250000      0.500000
5         India          5          0.600000      0.600000
6         Italy          2          0.500000      0.500000
7         Japan         17          0.647059      0.647059
8         Korea          3          1.000000      1.000000
9        Mexico          1          0.000000      0.000000
10  Netherlands          1          0.000000      0.000000
11  South Korea          2          0.000000      0.500000
12  Southern US          1          0.000000      0.000000
13     Thailand          3          1.000000      0.666667
14           UK          1          0.000000      0.000000
15           US         51 